# 2-RDM AIMD (Γ<sup>c</sup>)

NVE molecular dynamics for water at 300 K using a **machine-learned correlated 2-RDM** Γ<sup>c</sup>.

`db2qmmodel` fits `target='gamma2c'`. The ASE calculator uses `method='gamma2'`: it predicts Γ<sup>c</sup>, purifies it, reconstructs Γ, and evaluates the energy by contraction with one- and two-electron integrals.

Point `dbfile` at an HDF5 database that stores `gamma2c`. Production examples used 20 000 steps of 0.5 fs.


In [ ]:
import numpy as np
from ase.build import molecule
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, force_temperature
from ase.md.verlet import VelocityVerlet
from ase import units
from sklearn.kernel_ridge import KernelRidge

from qmlearn.io.model import db2qmmodel
from qmlearn.api.api4ase import QMLCalculator

In [ ]:
dbfile = "train.hdf5"

T = 300
nsteps = 200
timestep = 0.5 * units.fs
np.random.seed(8888)

In [ ]:
models = {"gamma2c": KernelRidge(alpha=0.0, kernel="rbf")}

qmmodel = db2qmmodel(
    dbfile,
    names="*",
    mmodels=models,
    target="gamma2c",
    method="gamma2c",
    purify_gamma=True,
)

In [ ]:
atoms = molecule("H2O")
atoms.calc = QMLCalculator(
    qmmodel=qmmodel,
    method="gamma2",
    properties=("energy",),
)

MaxwellBoltzmannDistribution(atoms, temperature_K=T, force_temp=True)
p = atoms.get_momenta()
p -= p.sum(axis=0) / len(atoms)
atoms.set_momenta(p)
force_temperature(atoms, T)

dyn = VelocityVerlet(
    atoms,
    timestep=timestep,
    trajectory="md_nve_gamma2c.traj",
    logfile="md_nve_gamma2c.log",
)
dyn.run(nsteps)
print("Finished", nsteps, "steps; energy =", atoms.get_potential_energy())